# Subproyecto 2 — piloto con RAG

Compara GPT-4o mini, LLaMA 3.1 8B y Mistral con recuperación aumentada. El corpus de conocimiento contiene tres documentos por intención y excluye coincidencias exactas y cercanas con las 20 consultas de evaluación.

## Protocolo

- Embeddings: `paraphrase-multilingual-MiniLM-L12-v2`.
- Condición principal: conocimiento original en inglés; la traducción al español se conserva como sensibilidad (rindió peor en el piloto).
- Búsqueda coseno exacta con `top_k=3`.
- El contexto recuperado es idéntico para los tres modelos.
- Se registran por separado latencia de recuperación, generación y total.
- Los modelos locales se precalientan antes de medir.
- La ejecución generativa comienza desactivada para evitar llamadas pagas accidentales.

In [ ]:
import json, os, sys, time
from datetime import datetime
from pathlib import Path

sys.path.insert(0, '..')
os.environ.setdefault('HF_HOME', '../.cache/huggingface')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
from dotenv import load_dotenv
load_dotenv('../.env')
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from src import framework_tf as ftf
from src.rag import IndiceRAG

TOP_K = 3
IDIOMA_CONOCIMIENTO = 'en'  # Configuración primaria fijada después del análisis piloto.
EJECUTAR_EXPERIMENTO = False  # Cambiar a True solo para correr 60 generaciones.

## 1. Carga del corpus y construcción del índice

In [ ]:
evaluacion = pd.read_csv('../data/corpus_es_muestra.csv')
rutas_corpus = {
    'en': Path('../data/conocimiento_rag.csv'),
    'es': Path('../data/conocimiento_rag_es.csv'),
}
ruta_corpus = rutas_corpus[IDIOMA_CONOCIMIENTO]
if not ruta_corpus.exists():
    raise FileNotFoundError(f'No existe el corpus seleccionado: {ruta_corpus}')
conocimiento = pd.read_csv(ruta_corpus)

print(f'Corpus primario: {ruta_corpus.name} — {len(conocimiento)} documentos, {conocimiento.intent.nunique()} intenciones')
indice = IndiceRAG(conocimiento).construir()
print('Índice RAG listo')

## 2. Validación de recuperación

In [ ]:
recuperaciones = {}
filas_recuperacion = []
for idx, fila in evaluacion.iterrows():
    salida = indice.recuperar(fila['instruction_es'], top_k=TOP_K)
    recuperaciones[idx] = salida
    intents = [doc['intent'] for doc in salida['documentos']]
    filas_recuperacion.append({
        'consulta': fila['instruction_es'],
        'intent_esperado': fila['intent'],
        'intent_top1': intents[0],
        'hit_top1': intents[0] == fila['intent'],
        'hit_top_k': fila['intent'] in intents,
        'latencia_recuperacion_s': salida['latencia_recuperacion_s'],
    })
validacion = pd.DataFrame(filas_recuperacion)
print(f'Hit@1: {validacion.hit_top1.mean():.1%}')
print(f'Hit@{TOP_K}: {validacion.hit_top_k.mean():.1%}')
print(f'Latencia media: {validacion.latencia_recuperacion_s.mean() * 1000:.2f} ms')
validacion[~validacion.hit_top_k]

## 3. Generación con el mismo contexto para los tres modelos

In [ ]:
def precalentar_modelos_locales():
    ftf.verificar_ollama()
    for modelo in ftf.MODELOS:
        if ftf.MODELOS[modelo]['proveedor'] == 'ollama':
            salida = ftf.generar(modelo, 'Respondé solamente: listo')
            if salida['error']:
                raise RuntimeError(f'Falló el precalentamiento de {modelo}: {salida["error"]}')
            print(f'{modelo}: precalentado')

def correr_modelo_rag(modelo_key):
    filas = []
    for idx, fila in tqdm(evaluacion.iterrows(), total=len(evaluacion), desc=modelo_key):
        rec = recuperaciones[idx]
        salida = ftf.generar(modelo_key, fila['instruction_es'], contexto=rec['contexto'])
        latencia_generacion = salida['latencia_s']
        filas.append({
            'modelo': modelo_key,
            'tipo': ftf.MODELOS[modelo_key]['tipo'],
            'condicion': 'rag',
            'category': fila['category'],
            'intent': fila['intent'],
            'consulta': fila['instruction_es'],
            'referencia': fila['response_es'],
            'documentos_recuperados': json.dumps([d['document_id'] for d in rec['documentos']]),
            'scores_recuperacion': json.dumps([round(d['score'], 6) for d in rec['documentos']]),
            'latencia_recuperacion_s': rec['latencia_recuperacion_s'],
            'latencia_generacion_s': latencia_generacion,
            'latencia_total_s': round(rec['latencia_recuperacion_s'] + latencia_generacion, 6),
            **salida,
        })
        if ftf.MODELOS[modelo_key]['proveedor'] == 'openai':
            time.sleep(0.5)
    return pd.DataFrame(filas)

In [ ]:
RUTA_RAW = Path('../resultados_rag_piloto.csv')
RUTA_EVALUADOS = Path('../resultados_rag_piloto_evaluados.csv')
resultados = pd.DataFrame()
resultados_evaluados = pd.DataFrame()
if EJECUTAR_EXPERIMENTO:
    precalentar_modelos_locales()
    resultados = pd.concat(
        [correr_modelo_rag(modelo) for modelo in ftf.MODELOS],
        ignore_index=True,
    )
    resultados.to_csv(RUTA_RAW, index=False)
    print(f'Completadas: {resultados.error.isna().sum()}/{len(resultados)}')
elif RUTA_EVALUADOS.exists():
    resultados_evaluados = pd.read_csv(RUTA_EVALUADOS)
    print(f'Resultados evaluados reutilizados: {len(resultados_evaluados)} filas')
elif RUTA_RAW.exists():
    resultados = pd.read_csv(RUTA_RAW)
    print(f'Resultados crudos reutilizados: {len(resultados)} filas')
else:
    print('Sin resultados previos. Para generar, cambiá EJECUTAR_EXPERIMENTO a True.')

## 4. Calidad, costos y guardado

In [ ]:
if not resultados_evaluados.empty:
    ok = resultados_evaluados.copy()
    display(ok.groupby('modelo')[['bertscore_f1', 'rouge_l', 'latencia_recuperacion_s', 'latencia_generacion_s', 'latencia_total_s', 'costo_oficial_usd']].mean().round(6))
elif not resultados.empty:
    from bert_score import score as bertscore
    from rouge_score import rouge_scorer

    ok = resultados[resultados.error.isna()].copy()
    _, _, f1 = bertscore(cands=ok.respuesta.tolist(), refs=ok.referencia.tolist(), lang='es', verbose=False)
    ok['bertscore_f1'] = f1.numpy().round(4)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    ok['rouge_l'] = [round(scorer.score(ref, cand)['rougeL'].fmeasure, 4) for ref, cand in zip(ok.referencia, ok.respuesta)]
    costos = ok.apply(lambda fila: pd.Series(ftf.costo_consulta(fila)), axis=1)
    ok = pd.concat([ok, costos], axis=1)
    ok['costo_oficial_usd'] = ok.apply(ftf.costo_oficial, axis=1)
    display(ok.groupby('modelo')[['bertscore_f1', 'rouge_l', 'latencia_recuperacion_s', 'latencia_generacion_s', 'latencia_total_s', 'costo_oficial_usd']].mean().round(6))

    ok.to_csv(RUTA_EVALUADOS, index=False)
    print(f'Guardado: {RUTA_EVALUADOS} ({len(ok)} filas)')
else:
    print('Sin resultados generativos para evaluar.')